# RS/OF tuning orientation by cortical layer — FENS 2026

Analyses the orientation and eccentricity of 2D Gaussian tuning in the RS/OF plane,
split by cortical layer (Layer 2/3 vs Layer 5).

**Inputs:** precomputed `neurons_df_trial_average.pickle` (2D Gaussian fits already stored).  
**No preprocessing is performed** — all model fits are read directly from the pickle.  
**Outputs:** BIC/AIC model-selection curves and rose plots of tuning orientation per layer,
with von Mises mixture density overlaid.

In [ ]:
%load_ext autoreload
%autoreload 2
import re
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib
from scipy.stats import norm
import flexiznam as flz
import cottage_analysis.analysis.fit_gaussian_blob as fit_gb
from v1_depth_map.figure_utils import von_mises
from v1_depth_map.figure_utils.von_mises import axial_mixture_density

import matplotlib.font_manager as fm

# Style
sns.set_theme(style="ticks", context="talk")
font_path = '/nemo/lab/znamenskiyp/home/shared/resources/fonts/arial.ttf'
fm.fontManager.addfont(font_path)
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = 'Arial'


def min_sigma_from_col(col):
    """Recover the min_sigma a fit was run with from its column-name tag.

    The trial-average pipeline tags non-default min_sigma fits as ``_ms<value>``
    just before ``_treadmill`` (e.g. ``_ms0p1`` -> 0.1, ``_ms0`` -> 0.0), with
    ``p`` for the decimal point and ``m`` for a minus sign. Untagged columns use
    the pipeline default of 0.25. Always feed this value to the get_gaussian_*/
    get_semi*_length / convert_* helpers so they reconstruct the same variance
    floor the fit used.
    """
    m = re.search(r"_ms([0-9pm]+?)_treadmill", col)
    if m is None:
        return 0.25
    return float(m.group(1).replace("p", ".").replace("m", "-"))

## Parameters

- `PROJECTS`: the two flexilims projects containing the sessions of interest.
- `LAYER_CUTOFF`: nominal recording depth (µm) dividing Layer 2/3 from Layer 5.
- `ECCENTRICITY_CUTOFF`: minimum ellipse eccentricity (0–1) — keeps only neurons with clearly elongated tuning.
- `PERCENTILE`: percentile of the empirical null R² distribution used as significance threshold.
- `BEST_MODEL_ONLY`: if `True`, restrict to neurons whose best-fitting RS/OF model is the 2D Gaussian (g2d).
- `N_ANGLE_BINS`: number of angular bins in the rose plot (10° bins with 18 bins over 180°).
- `POPT_FORMAT`: `"legacy"` for fits made with the old (σx, σy, θ) parameterisation (converted to Cholesky on load), or `"cholesky"` for data re-fit with the new precision-matrix parameterisation (no conversion).

In [ ]:
PROJECTS = ["colasa_3d-vision_revisions", "ccyp_l5_3d_vision"]
NEURONS_DF_PATH = "/camp/home/blota/code/v1_depth_map/neurons_df_trial_average.pickle"
LAYER_CUTOFF = 350        # µm — Layer 2/3 ≤ 350, Layer 5 > 350
ECCENTRICITY_CUTOFF = 0.4 # keep only elongated tuning ellipses
SIGMA_MAJOR_CUTOFF = 5.0  # semi-major σ (log RS/OF units) above this = unconstrained
                          # "ridge": the Gaussian is flat across the whole sampled plane,
                          # i.e. effectively 1D and its elongation is not data-constrained
PERCENTILE = 95           # percentile of empirical null for RSQ significance
BEST_MODEL_ONLY = False    # restrict to neurons whose best model is g2d
N_ANGLE_BINS = 18       # 180° / 18 = 10° per bin

POPT_FORMAT = "cholesky"    # "legacy" = old (sigma_x, sigma_y, theta) fits;
                          # "cholesky" = re-fit with the new parameterisation

# The g2d fit used for all ellipse geometry (angle / eccentricity / semi-axis sigmas).
# MIN_SIGMA is recovered from the column's _ms tag so every helper reconstructs the
# same variance floor the fit was run with.
popt_col = "rsof_popt_closedloop_g2d_treadmill_trial_average"
MIN_SIGMA = min_sigma_from_col(popt_col)
print(f"popt_col={popt_col}\n  -> min_sigma={MIN_SIGMA}")


## Load data

Load the precomputed `neurons_df` from pickle, then query flexilims for session metadata
(specifically `nominal_depth`) from both projects. The nominal depth is mapped onto every
neuron and used to assign a layer label.

In [ ]:
neurons_df = pd.read_pickle(NEURONS_DF_PATH)
neurons_df["is_depth_neuron"] = (
    (neurons_df["depth_tuning_test_spearmanr_rval_closedloop"] > 0.1)
    & (neurons_df["depth_tuning_test_spearmanr_pval_closedloop"] < 0.05)
)
neurons_df["is_depth_neuron_treadmill"] = (
    (neurons_df["depth_tuning_test_spearmanr_rval_closedloop_treadmill"] > 0.1)
    & (neurons_df["depth_tuning_test_spearmanr_pval_closedloop_treadmill"] < 0.05)
)
print(f"{len(neurons_df)} neurons from {neurons_df.session.nunique()} sessions")

In [ ]:
str(NEURONS_DF_PATH)

In [ ]:
from cottage_analysis.analysis import fit_gaussian_blob

neurons_df['g2d_semiminor_length'] = neurons_df[popt_col].apply(lambda x: fit_gaussian_blob.get_semiminor_length(x, min_sigma=MIN_SIGMA))
neurons_df['g2d_semimajor_length'] = neurons_df[popt_col].apply(lambda x: fit_gaussian_blob.get_semimajor_length(x, min_sigma=MIN_SIGMA))

In [ ]:
all_sessions_dfs = []
for project in PROJECTS:
    flm_sess = flz.get_flexilims_session(project_id=project)
    sessions = flz.get_entities(datatype="session", flexilims_session=flm_sess)
    all_sessions_dfs.append(sessions)
all_sessions_df = pd.concat(all_sessions_dfs).copy()

# nominal_depth may be stored as a list/array in flexilims — take the mean
all_sessions_df["nominal_depth"] = all_sessions_df["nominal_depth"].apply(
    lambda x: np.mean(x) if hasattr(x, "__len__") else x
)

neurons_df["nominal_depth"] = neurons_df["session"].map(all_sessions_df["nominal_depth"])
neurons_df["layer"] = neurons_df["nominal_depth"].apply(
    lambda d: "Layer 2/3" if d <= LAYER_CUTOFF else "Layer 5"
)
neurons_df = neurons_df.copy()
print(neurons_df["layer"].value_counts())
print(f"No nominal_depth: {neurons_df['nominal_depth'].isna().sum()} neurons")

## Ellipse properties

Each neuron's 2D Gaussian fit (stored in `rsof_popt_closedloop_g2d_treadmill`) encodes the
orientation and shape of its RS/OF tuning as an ellipse. Here we derive:

- **theta** (`g2d_theta_treadmill`): orientation angle of the ellipse major axis (degrees).
  An angle of 0° means tuning aligned with the OF axis; 90° means RS axis.
- **eccentricity** (`g2d_eccentricity_treadmill`): 1 − minor/major axis ratio. 0 = circular, 1 = maximally elongated.

In [ ]:
# Legacy fits store (log_sigma_x2, log_sigma_y2, theta); the current
# get_gaussian_angle / get_gaussian_eccentricity expect the Cholesky
# (log_l11, l21, log_l22) format. Convert legacy fits before extracting below.
# popt_col / MIN_SIGMA are defined in the parameters cell.
if POPT_FORMAT == "legacy":
    from cottage_analysis.analysis.fit_gaussian_blob import (
        convert_g2d_popt_old_to_cholesky,
    )
    ok = neurons_df[popt_col].apply(
        lambda x: isinstance(x, (list, np.ndarray)) and len(x) >= 7
    )
    neurons_df.loc[ok, popt_col] = neurons_df.loc[ok, popt_col].apply(
        lambda x: convert_g2d_popt_old_to_cholesky(x, min_sigma=MIN_SIGMA)
    )
    print(f"Converted {ok.sum()} legacy popt arrays to Cholesky parameterisation.")
elif POPT_FORMAT == "cholesky":
    print("Popt already in Cholesky format — no conversion needed.")
else:
    raise ValueError(f"Unknown POPT_FORMAT: {POPT_FORMAT!r}")

In [ ]:
ok = neurons_df[popt_col].notna()
neurons_df.loc[ok, "g2d_theta_treadmill"] = neurons_df.loc[ok, popt_col].apply(
    lambda x: fit_gb.get_gaussian_angle(x, min_sigma=MIN_SIGMA)
)
neurons_df.loc[ok, "g2d_eccentricity_treadmill"] = neurons_df.loc[ok, popt_col].apply(
    lambda x: fit_gb.get_gaussian_eccentricity(x, min_sigma=MIN_SIGMA)
)
print(f"Ellipse properties computed for {ok.sum()} neurons")

## Significance thresholds and neuron filtering

Rather than using a hard-coded flag, RS/OF significance is determined from the data itself:

1. **Empirical null**: for each model, the negative tail of the test R² distribution is
   assumed to be uncontaminated by real signal. Its standard deviation estimates the null
   width; the `PERCENTILE`-th percentile of N(0, σ) is the threshold.
2. **Depth tuning**: neuron must be depth-selective (Spearman r > 0.1, p < 0.05 in the
   spheres closed-loop condition; stored in `is_depth_neuron`).
3. **Eccentricity**: ellipse eccentricity > `ECCENTRICITY_CUTOFF` (default 0.4).
4. **Best model** (optional): if `BEST_MODEL_ONLY=True`, only neurons for which the g2d
   (2D Gaussian) achieves the highest test R² across all seven models are kept.

In [ ]:
def empirical_null_threshold(rsq_values, percentile=95):
    """Threshold from the negative tail of the RSQ distribution (empirical null)."""
    vals = np.asarray(rsq_values, dtype=float)
    vals = vals[np.isfinite(vals)]
    neg = vals[vals < 0]
    if len(neg) < 20:
        raise ValueError(f"Only {len(neg)} negative values — too few to fit null reliably")
    sigma = np.std(neg)
    threshold = norm.ppf(percentile / 100, loc=0, scale=sigma)
    return threshold, sigma


models = [
    "gof_treadmill", "grs_treadmill", "gadd_treadmill",
    "g2d_treadmill", "gratio_treadmill", "g2mult_treadmill", "gprod_treadmill",
]
model_rsq_cols = {m: f"rsof_test_rsq_closedloop_{m}_trial_average" for m in models}

thresholds = {}
for model, col in model_rsq_cols.items():
    vals = neurons_df[col].dropna().values
    vals = vals[np.isfinite(vals) & (vals >= -1)]  # drop sentinel values
    thr, sigma = empirical_null_threshold(vals, percentile=PERCENTILE)
    thresholds[model] = thr
    neurons_df[f"{col}_sig"] = neurons_df[col] > thr
    print(
        f"{model:30s}  sigma={sigma:.4f}  threshold={thr:.4f}  "
        f"passing={neurons_df[f'{col}_sig'].mean() * 100:.1f}%"
    )

# Best model per neuron: column name with highest test RSQ
neurons_df["best_model"] = neurons_df[list(model_rsq_cols.values())].idxmax(axis=1)

In [ ]:
g2d_col = model_rsq_cols["g2d_treadmill"]

mask = (
    neurons_df[f"{g2d_col}_sig"]                          # g2d RSQ above empirical null threshold
    & neurons_df["is_depth_neuron_treadmill"].fillna(False)         # depth-tuned (Spearman r>0.1, p<0.05)
    & (neurons_df["g2d_eccentricity_treadmill"] > ECCENTRICITY_CUTOFF)
)
if BEST_MODEL_ONLY:
    mask = mask & (neurons_df["best_model"] == g2d_col)

ndf = neurons_df[mask].copy()
print(f"{len(ndf)} neurons pass all filters ({mask.mean():.1%} of total)")
print(ndf["layer"].value_counts())
print(len(ndf.session.unique()))

# Example cell

In [ ]:
EXAMPLE_SESSION = "BRAC12025.1g_S20251212"
EXAMPLE_PROJECT ="ccyp_l5_3d_vision"

In [ ]:
# Load ndf, trials_df_tm, trials_df_tm_no_cut and trials_df_sphere

from cottage_analysis.pipelines import pipeline_utils
from cottage_analysis.plotting.rsof_plots import plot_RS_OF_matrix, plot_RS_OF_fit

example_mouse, example_session = EXAMPLE_SESSION.split('_')

print(f"Loading data for example session: {example_session}, mouse: {example_mouse}")

# Load datasets using helper function
ndf_example, trials_df_tm, trials_df_sphere = pipeline_utils.load_treadmill_and_sphere_datasets(
    EXAMPLE_PROJECT,
    example_mouse,
    example_session,
    photodiode_protocol=5,
    filter_datasets={"anatomical_only": 3, "annotated": True},
    recording_type="two_photon",
    protocol_base_sphere="SpheresPermTubeReward",
)
# Load a version of trials_df_tm without cuttin the ramp
_, trials_df_tm_no_cut, _ = pipeline_utils.load_treadmill_and_sphere_datasets(
    EXAMPLE_PROJECT,
    example_mouse,
    example_session,
    photodiode_protocol=5,
    filter_datasets={"anatomical_only": 3, "annotated": True},
    recording_type="two_photon",
    protocol_base_sphere="SpheresPermTubeReward",
    tread_kwargs=dict(   
        cut_trial_end=None,
        trial_duration=None,
        acceleration_time=None)
)
# Find frame rate
suite2p_ds = flz.get_datasets(
    origin_name=f"{example_mouse}_{example_session}", 
    dataset_type='suite2p_rois',
    filter_datasets=dict(annotated=True),
    flexilims_session=flz.get_flexilims_session(project_id=EXAMPLE_PROJECT),
    allow_multiple=False)
fs = suite2p_ds.extra_attributes['fs']


In [ ]:
expected_of = trials_df_tm.expected_optic_flow_stim.map(np.nanmedian)
expected_rs = trials_df_tm.MotorSpeed_stim.map(np.nanmedian)
df = pd.DataFrame({"expected_of": expected_of, "expected_rs": expected_rs, "depth": trials_df_tm.depth}) 

## Von Mises mixture model

The distribution of tuning orientations is axial (period π, not 2π). We fit a mixture of
von Mises distributions (k = 1 … 5 components) separately for each layer and select the
best k by BIC. The EM fitting is handled by `von_mises.model_selection()` in
`v1_depth_map.figure_utils.von_mises`.

In [ ]:
results = {}
for layer, group in ndf.groupby("layer"):
    print(layer)
    thetas = np.radians(group["g2d_theta_treadmill"].dropna().values)
    ms = von_mises.model_selection(thetas, max_k=5, seed=42)
    best_k = np.argmin([ms[k]["bic"] for k in sorted(ms)]) + 1
    results[layer] = {"model_selection": ms, "best_k": best_k, "thetas": thetas}
    print(f"{layer}: n={len(thetas)}, best k={best_k}")

## BIC / AIC model selection curves

Lower BIC/AIC indicates a better model. The selected k is the one that minimises BIC.

In [ ]:
for layer, res in results.items():
    fig, (ax_c, ax_h) = plt.subplots(1, 2, figsize=(14, 5))
    von_mises.plot_model_selection(
        res["thetas"],
        res["model_selection"],
        ax_criteria=ax_c,
        ax_hist=ax_h,
    )
    fig.suptitle(f"{layer} (n={len(res['thetas'])}), best k={res['best_k']}")
    plt.tight_layout()
    plt.show()

# GMM version

Same but with gaussian as there is not a single cell in the -40 - 120 degrees segment

In [ ]:
from sklearn.mixture import GaussianMixture

gmm_results = {}
for layer, group in ndf.groupby("layer"):
    print(layer)
    # GMM is a linear model, so fit on the angles directly (in degrees).
    angles = group["g2d_theta_treadmill"].dropna().values.reshape(-1, 1)
    n = len(angles)

    ms = {}
    for k in range(1, 5):  # matches max_k=5 above
        gm = GaussianMixture(
            n_components=k,
            covariance_type="full",
            n_init=10,
            random_state=42,
        ).fit(angles)
        ll = gm.score(angles) * n  # total log-likelihood (score is the mean)
        ms[k] = {
            "bic": gm.bic(angles),
            "aic": gm.aic(angles),
            "ll": ll,
            "model": gm,
        }
        means_deg = np.sort(gm.means_.ravel()).round(1)
        weights = gm.weights_[np.argsort(gm.means_.ravel())].round(3)
        print(
            f"k={k}  ll={ll:9.1f}  BIC={ms[k]['bic']:9.1f}  AIC={ms[k]['aic']:9.1f}  "
            f"means={means_deg} \u00b0  weights={weights}"
        )

    best_bic = min(ms, key=lambda k: ms[k]["bic"])
    best_aic = min(ms, key=lambda k: ms[k]["aic"])
    print(f"\nBest k by BIC: {best_bic}")
    print(f"Best k by AIC: {best_aic}\n")

    gmm_results[layer] = {"model_selection": ms, "best_k": best_bic, "angles": angles.ravel()}


## GMM model selection

BIC and AIC as a function of the number of components `k` for the classic Gaussian mixture (fit on the raw orientation angles in degrees), with the best-BIC and best-AIC densities overlaid on the histogram — the linear-model counterpart to the von Mises selection figure above.

In [ ]:
# BIC/AIC model-selection curves for the GMM (mirrors von_mises.plot_model_selection).
angle_lo, angle_hi = -45, 135
grid_deg = np.linspace(angle_lo, angle_hi, 600)
bins = np.arange(angle_lo, angle_hi + 5, 5)
col_bic, col_aic = "steelblue", "darkorange"

for layer, res in gmm_results.items():
    ms = res["model_selection"]
    ks = sorted(ms)
    best_bic = min(ks, key=lambda k: ms[k]["bic"])
    best_aic = min(ks, key=lambda k: ms[k]["aic"])

    fig, (ax_c, ax_h) = plt.subplots(1, 2, figsize=(14, 5))

    # --- Left: BIC & AIC vs k ---
    ax_c.plot(ks, [ms[k]["bic"] for k in ks], "o-", color=col_bic, label="BIC")
    ax_c.plot(ks, [ms[k]["aic"] for k in ks], "s--", color=col_aic, label="AIC")
    ax_c.axvline(best_bic, color=col_bic, alpha=0.35, ls=":")
    ax_c.axvline(best_aic, color=col_aic, alpha=0.35, ls=":")
    ax_c.set_xlabel("Number of components k")
    ax_c.set_ylabel("Information criterion")
    ax_c.set_title("GMM model selection (lower = better)")
    ax_c.set_xticks(ks)
    ax_c.legend()

    # --- Right: histogram + best-BIC / best-AIC densities ---
    ax_h.hist(
        res["angles"], bins=bins, density=True,
        color="lightgray", edgecolor="white", alpha=0.85, label="Data",
    )
    for best_k, ls, col, prefix in [
        (best_bic, "-", col_bic, "BIC"),
        (best_aic, "--", col_aic, "AIC"),
    ]:
        gm = ms[best_k]["model"]
        dens = np.exp(gm.score_samples(grid_deg.reshape(-1, 1)))  # already per-degree
        ax_h.plot(grid_deg, dens, ls=ls, color=col, lw=2,
                  label=f"{prefix} best k={best_k}")
        for mu_j in gm.means_.ravel():
            ax_h.axvline(mu_j, color=col, alpha=0.35, ls=":")

    ax_h.set_xlim(angle_lo, angle_hi)
    ax_h.set_xlabel("Orientation (degrees)")
    ax_h.set_ylabel("Density")
    ax_h.set_title("GMM mixture - best models")
    ax_h.legend(fontsize=8)

    fig.suptitle(f"{layer} (n={len(res['angles'])}), best k={res['best_k']}")
    plt.tight_layout()
    plt.show()


In [ ]:
# Histograms for layer 2/3 and layer 5 side by side, with best-BIC GMM mixture densities overlaid.
angle_lo, angle_hi = -45, 135
grid_deg = np.linspace(angle_lo, angle_hi, 600)
bins = np.arange(angle_lo, angle_hi + 5, 5)
layer_colors = {"Layer 2/3": "steelblue", "Layer 5": "crimson"}

layers = list(gmm_results)
fig, axes = plt.subplots(1, len(layers), figsize=(4 * len(layers), 3), sharey=True)
if len(layers) == 1:
    axes = [axes]

for ax, layer in zip(axes, layers):
    res = gmm_results[layer]
    col = layer_colors.get(layer, None)
    n = len(res["angles"])

    ax.hist(
        res["angles"], bins=bins, density=True,
        color=col, edgecolor='None', alpha=0.3,
        label=f"cells (n={n})",
    )

    gm = res["model_selection"][res["best_k"]]["model"]
    dens = np.exp(gm.score_samples(grid_deg.reshape(-1, 1)))  # already per-degree

    # Describe the fitted mixture in the legend: number of components and peak
    # locations (component means), ordered by orientation.
    order = np.argsort(gm.means_.ravel())
    peaks = gm.means_.ravel()[order]
    weights = gm.weights_[order]
    peak_str = ", ".join(f"{m:.0f}\u00b0 ({w:.0%})" for m, w in zip(peaks, weights))
    gmm_label = f"GMM (k={res['best_k']})\npeaks: {peak_str}"

    ax.plot(grid_deg, dens, color=col, lw=2, alpha=0.7, label=gmm_label)

    ax.set_xlim(angle_lo, angle_hi)
    ax.set_xlabel("Orientation (degrees)")
    ax.set_xticks([0, 45, 90])
    ax.set_title(layer)
    ax.legend(fontsize=7, frameon=False, loc="upper right")

axes[0].set_ylabel("Density")
sns.despine()
plt.tight_layout()
plt.savefig("rsof_angles_by_layer_gmm.pdf", bbox_inches="tight")
plt.show()

## Scatter plot — eccentricity vs angle by layer

Each neuron is shown as a point in polar coordinates: angular position = tuning orientation,
radial position = ellipse eccentricity (0 = circular, 1 = maximally elongated).
Neurons pass the g2d RSQ significance threshold and depth-tuning criterion; **no eccentricity
lower bound is applied** so the full range (0–1) is visible. The von Mises mixture density
fitted on the `ndf` subset (eccentricity > `ECCENTRICITY_CUTOFF`) is overlaid.

In [ ]:
[c for c in neurons_df.columns if 'minor' in c]

In [ ]:
theta_range = np.linspace(-np.pi / 2, np.pi / 2, 500)

# Use the GMM fit results when available (the von Mises fit is degenerate here because
# some layers have no cells across part of the angular range); fall back to `results`.
layer_results = gmm_results if "gmm_results" in globals() else results
layers = list(layer_results)

# Scatter uses all eccentricities — no lower bound, only g2d significance + depth tuning
mask_scatter = (
    neurons_df[f"{g2d_col}_sig"]
    & neurons_df["is_depth_neuron_treadmill"].fillna(False)
)
if BEST_MODEL_ONLY:
    mask_scatter = mask_scatter & (neurons_df["best_model"] == g2d_col)
ndf_scatter = neurons_df[mask_scatter].copy()

fig, axes = plt.subplots(
    1, len(layers), subplot_kw={"projection": "polar"}, figsize=(5 * len(layers), 5)
)
axes = np.atleast_1d(axes)

for ax, layer in zip(axes, layers):
    group = ndf_scatter[ndf_scatter["layer"] == layer]
    thetas = np.radians(group["g2d_theta_treadmill"].values)
    ecc = group["g2d_eccentricity_treadmill"].values

    sc = ax.scatter(thetas, ecc, c=group["g2d_semiminor_length"],
                     s=30, alpha=0.3, linewidths=0, vmax=2)

    ax.set_title(f"{layer} (n={len(thetas)})")
    ax.set_ylim(0, 1)
    ax.set_ylabel("eccentricity", labelpad=30)
    ax.set_theta_zero_location('E')  # 0 is East (Right) -> OF
    ax.set_thetalim(np.radians(-45), np.radians(135))        # Show -45 to 135 degrees
    ax.set_xticks(np.radians([-45, 0, 45, 90, 135]))
    ax.set_xticklabels(['-45°', '0°', '45°', '90°', '135°'])

plt.colorbar(sc, ax=axes, label="g2d semiminor length (σ_min)", orientation="vertical", fraction=0.02, pad=0.04)
plt.savefig("rsof_scatter_by_layer.pdf", bbox_inches="tight")
plt.show()


In [ ]:
theta_range = np.linspace(-np.pi / 2, np.pi / 2, 500)

# Scatter uses all eccentricities — no lower bound, only g2d significance + depth tuning
mask_scatter = (
    neurons_df[f"{g2d_col}_sig"]
    & neurons_df["is_depth_neuron_treadmill"].fillna(False)
)
if BEST_MODEL_ONLY:
    mask_scatter = mask_scatter & (neurons_df["best_model"] == g2d_col)
ndf_scatter = neurons_df[mask_scatter].copy()

fig, axes = plt.subplots(
    1, len(results), subplot_kw={"projection": "polar"}, figsize=(5 * len(results), 5)
)
if len(results) == 1:
    axes = [axes]

for ax, (layer, res) in zip(axes, results.items()):
    group = ndf_scatter[ndf_scatter["layer"] == layer]
    thetas = np.radians(group["g2d_theta_treadmill"].values)
    ecc = group["g2d_eccentricity_treadmill"].values

    sc = ax.scatter(thetas, ecc, c=layer_colors[layer], s=30, alpha=0.3, linewidths=0)

    ax.set_title(f"{layer} (n={len(thetas)})")
    ax.set_ylim(0, 1)
    ax.set_ylabel("eccentricity", labelpad=30)
    ax.set_theta_zero_location('E')  # 0 is East (Right) -> OF
    ax.set_thetalim(np.radians(-45), np.radians(135))        # Show -45 to 135 degrees
    ax.set_xticks(np.radians([-45, 0, 45, 90, 135]))
    ax.set_xticklabels(['-45°', '0°', '45°', '90°', '135°'])


plt.savefig("rsof_scatter_by_layer.pdf", bbox_inches="tight")
plt.show()

## Rose plot — tuning angle distribution by layer

Polar histogram of the RS/OF tuning orientation (axial, −90° to 90°), one panel per layer.
Bar **radius = √n** so that bar *area* (∝ r²) is proportional to cell count — this avoids
the visual illusion where a doubling of count looks like a 4× increase.

The fitted von Mises mixture density (best k from model selection) is overlaid as a solid
black line; individual components are shown as dashed grey lines.

In [ ]:
bin_edges = np.linspace(-np.pi / 4, 3 * np.pi / 4, N_ANGLE_BINS + 1)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
bin_width = bin_edges[1] - bin_edges[0]
theta_range = np.linspace(-np.pi / 4, 3 * np.pi / 4, 500)

fig, axes = plt.subplots(
    1, len(results), subplot_kw={"projection": "polar"}, figsize=(5 * len(results), 5)
)
if len(results) == 1:
    axes = [axes]

for ax, (layer, res) in zip(axes, results.items()):
    thetas = (res["thetas"] + np.pi / 4) % np.pi - np.pi / 4
    counts, _ = np.histogram(thetas, bins=bin_edges)
    radii = np.sqrt(counts)  # sqrt(n): area of each wedge ∝ n

    ax.bar(bin_centers, radii, width=bin_width, align="center", alpha=0.6)

    # Von Mises density — normalise to match the sqrt(n) radial scale
    best = res["model_selection"][res["best_k"]]
    density = axial_mixture_density(
        theta_range, best["pi_k"], best["mu_k"], best["kappa_k"]
    )
    if True:
        # Scale: match the peak of the histogram (sqrt(n) scale)
        density_scaled = density /density.max() * radii.max()

        ax.plot(theta_range, density_scaled, color="k", linewidth=2, label="mixture")
        for pi_i, mu_i, kappa_i in zip(best["pi_k"], best["mu_k"], best["kappa_k"]):
            comp = axial_mixture_density(theta_range, [pi_i], [mu_i], [kappa_i])
            comp_scaled = comp / density.max() * radii.max()
            ax.plot(theta_range, comp_scaled, "--", color="gray", linewidth=1)

    ax.set_title(f"{layer} (n={len(thetas)})")
    ax.set_theta_zero_location('E')  # 0 is East (Right) -> OF
    ax.set_thetalim(np.radians(-45), np.radians(135))        # Show -45 to 135 degrees
    ax.set_xticks(np.radians([-45, 0, 45, 90, 135]))
    ax.set_xticklabels(['-45°', '0°', '45°', '90°', '135°'])

plt.tight_layout()
plt.savefig("rsof_angles_by_layer.pdf", bbox_inches="tight")
plt.show()

In [ ]:
totbysess = neurons_df[neurons_df.layer=='Layer 2/3'].session.value_counts()
fitbysess = neurons_df[(neurons_df.layer=='Layer 2/3') & neurons_df[f"{g2d_col}_sig"]].session.value_counts()
depthbysess = neurons_df[(neurons_df.layer=='Layer 2/3') & neurons_df["is_depth_neuron_treadmill"]].session.value_counts()
bysessdf = pd.DataFrame(dict(total=totbysess, g2d=fitbysess, depth=depthbysess))

In [ ]:
bysessdf

In [ ]:
# In[6b]: semi-axis widths (Gaussian sigma along each ellipse axis)
ok = neurons_df[popt_col].notna()
neurons_df.loc[ok, "g2d_sigma_major_treadmill"] = neurons_df.loc[ok, popt_col].apply(
    lambda x: fit_gb.get_semimajor_length(x, min_sigma=MIN_SIGMA)
)
neurons_df.loc[ok, "g2d_sigma_minor_treadmill"] = neurons_df.loc[ok, popt_col].apply(
    lambda x: fit_gb.get_semiminor_length(x, min_sigma=MIN_SIGMA)
)

In [ ]:
[c for c in neurons_df.columns if 'sigma_minor' in c and 'g2d' in c]

In [ ]:
vals

In [ ]:
import seaborn as sns

fig, axes = plt.subplots(1,2, figsize=(7, 4))
vals = neurons_df.loc[mask, ["layer", "g2d_sigma_minor_treadmill", "g2d_sigma_major_treadmill"]].dropna()
ok = vals[~vals.isna().any(axis=1)]
kw = dict(alpha=0.6, common_norm=False, 
          stat="density", element="step", fill=False, hue='layer', log_scale=True, linewidth=1.5)
sns.histplot(data=ok, x="g2d_sigma_major_treadmill",
            ax=axes[0],  **kw)
sns.histplot(data=ok, x="g2d_sigma_minor_treadmill",
              ax=axes[1],  **kw)




In [ ]:
import seaborn as sns

fig, axes = plt.subplots(1,2, figsize=(7, 4))
vals = neurons_df.loc[mask, ["layer", "g2d_sigma_minor_treadmill", "g2d_sigma_major_treadmill"]].dropna()
ok = vals[~vals.isna().any(axis=1)]
kw = dict(alpha=0.6, common_norm=False, 
          stat="density", element="step", fill=False, hue='layer', log_scale=True, linewidth=1.5)
sns.histplot(data=ok, x="g2d_sigma_major_treadmill",
            ax=axes[0],  **kw)
sns.histplot(data=ok, x="g2d_sigma_minor_treadmill",
              ax=axes[1],  **kw)




In [ ]:
sns.scatterplot(data=ok, x="g2d_sigma_major_treadmill", y="g2d_sigma_minor_treadmill", hue='layer', alpha=0.6)

## Diagnostics: 1D-tuned cells vs genuine 2D tuning

Many cells pile up at exactly **0°** and **90°** — the cardinal OF / RS axes. These are largely
**1D-tuned** neurons (selective for optic flow *or* running speed alone): the 2D Gaussian then
stretches without bound along the un-tuned axis, snapping the major-axis angle to the cardinal
direction and saturating eccentricity near 1. (Convention: angle = direction of the ellipse
**major** axis; 90° → major axis along OF → cell is RS-selective / OF-invariant.)

Two independent signatures flag these cells:

- **Ridge** (`is_ridge`): semi-major σ exceeds `SIGMA_MAJOR_CUTOFF` (log RS/OF units) — the
  fitted Gaussian is essentially flat across the whole sampled plane, so its elongation is not
  constrained by the data.
- **1D model wins** (`rs_better` / `of_better`): a pure RS (`gRS`) or pure OF (`gOF`) Gaussian
  achieves a higher held-out R² than the 2D Gaussian (`g2d`).

**Plot A** splits the angle histogram into genuine-2D vs ridge. **Plot B** quantifies, per layer,
the fraction of cells better explained by gRS or gOF than by g2d. **Plot C** redraws angle vs
eccentricity after removing cells that a 1D model explains better.

In [ ]:
# Diagnostic columns: 1D-vs-2D model comparison + ridge flag.
# Model R² uses the standard min_sigma=0.25 fits (gRS/gOF were only fit there);
# ellipse geometry (angle/ecc/sigma) uses the popt_col fits as before.
g2d_rsq = neurons_df[model_rsq_cols["g2d_treadmill"]]
grs_rsq = neurons_df[model_rsq_cols["grs_treadmill"]]
gof_rsq = neurons_df[model_rsq_cols["gof_treadmill"]]
neurons_df["rs_better"] = grs_rsq > g2d_rsq          # pure-RS Gaussian beats g2d
neurons_df["of_better"] = gof_rsq > g2d_rsq          # pure-OF Gaussian beats g2d
neurons_df["either_better"] = neurons_df["rs_better"] | neurons_df["of_better"]
neurons_df["is_ridge"] = neurons_df["g2d_sigma_major_treadmill"] > SIGMA_MAJOR_CUTOFF

# Diagnostic population: g2d-significant + depth-tuned, NO eccentricity cutoff
# (so the full angle range and the ridges are visible).
diag_mask = (
    neurons_df[f"{g2d_col}_sig"]
    & neurons_df["is_depth_neuron_treadmill"].fillna(False)
)
if BEST_MODEL_ONLY:
    diag_mask = diag_mask & (neurons_df["best_model"] == g2d_col)
diag = neurons_df[diag_mask].copy()

print(f"{len(diag)} neurons in diagnostic population (g2d-sig + depth)")
print(f"  ridge (σ_major > {SIGMA_MAJOR_CUTOFF:g}): {diag['is_ridge'].mean() * 100:.1f}%")
print(f"  gRS beats g2d: {diag['rs_better'].mean() * 100:.1f}%   "
      f"gOF beats g2d: {diag['of_better'].mean() * 100:.1f}%   "
      f"either: {diag['either_better'].mean() * 100:.1f}%")

In [ ]:
# --- Plot A: angle distribution split into genuine-2D vs unconstrained ridge ---
layers = ["Layer 2/3", "Layer 5"]
bins = np.arange(-45, 136, 5)

fig, axes = plt.subplots(1, len(layers), figsize=(6 * len(layers), 4), sharey=True)
axes = np.atleast_1d(axes)
for ax, layer in zip(axes, layers):
    gdf = diag[diag["layer"] == layer]
    genuine = gdf.loc[~gdf["is_ridge"], "g2d_theta_treadmill"].dropna()
    ridge = gdf.loc[gdf["is_ridge"], "g2d_theta_treadmill"].dropna()
    ax.hist(
        [genuine, ridge], bins=bins, stacked=True, color=["#4477aa", "#cc6677"],
        label=[
            f"genuine 2D (σ_maj ≤ {SIGMA_MAJOR_CUTOFF:g})  n={len(genuine)}",
            f"ridge (σ_maj > {SIGMA_MAJOR_CUTOFF:g})  n={len(ridge)}",
        ],
    )
    for x in (0, 90):
        ax.axvline(x, color="k", ls=":", lw=1)
    ax.set_title(layer)
    ax.set_xlabel("g2d tuning angle (°)")
    ax.set_xticks([-45, 0, 45, 90, 135])
    ax.legend(fontsize=8)
axes[0].set_ylabel("neuron count")
fig.suptitle("Tuning-angle distribution: genuine 2D vs unconstrained ridge")
plt.tight_layout()
plt.show()

In [ ]:
# --- Plot B: proportion of cells better explained by a 1D model (gRS or gOF) than g2d, by layer ---
rows = []
for layer in layers:
    gdf = diag[diag["layer"] == layer]
    rows.append(dict(
        layer=layer, n=len(gdf),
        RS_better=gdf["rs_better"].mean(),
        OF_better=gdf["of_better"].mean(),
    ))
prop = pd.DataFrame(rows).set_index("layer")

fig, ax = plt.subplots(figsize=(6, 4))
# x = model (RS / OF), color = layer
plot_df = prop[["RS_better", "OF_better"]].mul(100).T
plot_df.index = ["gRS > g2d\n(RS-tuned)", "gOF > g2d\n(OF-tuned)"]
layer_colors = {"Layer 2/3": "#4477aa", "Layer 5": "#ee6677"}
plot_df.plot.bar(ax=ax, width=0.7, color=[layer_colors.get(l) for l in plot_df.columns])
ax.set_ylabel("% of cells where 1D model beats g2d")
ax.set_xlabel("")
ax.set_xticklabels(plot_df.index, rotation=0)
ax.legend([f"{l} (n={int(prop.loc[l, 'n'])})" for l in prop.index], title="layer")
ax.set_title("Cells better explained by a 1D Gaussian than g2d")
plt.tight_layout()
plt.show()

print(prop.assign(
    RS_better=lambda d: (d["RS_better"] * 100).round(1),
    OF_better=lambda d: (d["OF_better"] * 100).round(1),
).to_string())

In [ ]:
# --- Plot C: angle vs eccentricity after removing cells better explained by gRS or gOF ---
clean = diag[~diag["either_better"]].copy()
print(f"{len(clean)} cells remain after removing 1D-better cells "
      f"({len(clean) / len(diag) * 100:.0f}% of diagnostic population)")

fig, axes = plt.subplots(
    1, len(layers), subplot_kw={"projection": "polar"}, figsize=(5 * len(layers), 5)
)
axes = np.atleast_1d(axes)
for ax, layer in zip(axes, layers):
    group = clean[clean["layer"] == layer]
    thetas = np.radians(group["g2d_theta_treadmill"].values)
    ecc = group["g2d_eccentricity_treadmill"].values
    sc = ax.scatter(
        thetas, ecc, c='dodgerblue', s=25, alpha=0.4,
        linewidths=0, cmap="viridis", vmin=0, vmax=SIGMA_MAJOR_CUTOFF,
    )
    ax.set_title(f"{layer} (n={len(group)})")
    ax.set_ylim(0, 1)
    ax.set_ylabel("eccentricity", labelpad=30)
    ax.set_theta_zero_location("E")
    ax.set_thetalim(np.radians(-45), np.radians(135))
    ax.set_xticks(np.radians([-45, 0, 45, 90, 135]))
    ax.set_xticklabels(["-45°", "0°", "45°", "90°", "135°"])
plt.colorbar(sc, ax=axes, label="σ_major", fraction=0.02, pad=0.04)
fig.suptitle("Angle vs eccentricity — cells better fit by gRS/gOF removed")
plt.show()

## Running-speed (RS) tuned cells by layer

Does one layer have more RS-tuned cells, what do their tuning curves look like
(monotonically increasing / decreasing / peaked), and is that mix the same across layers?

- **RS-tuned**: the 1D running-speed Gaussian (`gRS`) is significant *and* has a higher
  held-out R² than **both** the optic-flow Gaussian (`gOF`) **and** the 2D Gaussian (`g2d`).
  Requiring `gRS > g2d` excludes cells with conjunctive RS×OF coding (product / 2D), which
  the 2D model captures better. `of_tuned` is defined symmetrically.
- **Tuning shape**: from the fitted gRS peak (`preferred_RS`) relative to the sampled running
  range `[RS_LO_CMS, RS_HI_CMS]` — peak **below** the range = *decreasing* (suppressed by
  running), **above** = *increasing* (monotonically rising), **inside** = *peaked* (a genuine
  preferred speed). The amplitude is always positive, so a monotonic response is fit as a
  Gaussian whose peak is pushed to / past the edge of the range; `preferred_RS` rails at the
  fit bounds (≈0.5 / 500 cm/s) for those cells.

Plots: **(1)** prevalence of RS- vs OF-tuned cells per layer; **(2)** the fitted gRS tuning
curves, normalised, split by shape and layer; **(3)** shape composition per layer with a χ²
test of whether the shape distribution differs between layers.

In [ ]:
# Identify RS-tuned cells and classify their tuning shape from the 1D-RS Gaussian (gRS).
from scipy.stats import chi2_contingency

RS_LO_CMS = 1.0    # peak below -> 'decreasing' over the sampled running range
RS_HI_CMS = 50.0   # peak above -> 'increasing'; in between -> 'peaked'

grs_popt_col = "rsof_popt_closedloop_grs_treadmill_trial_average"
grs_rsq_col = model_rsq_cols["grs_treadmill"]
gof_rsq_col = model_rsq_cols["gof_treadmill"]
g2d_rsq_col = model_rsq_cols["g2d_treadmill"]
GRS_MIN_SIGMA = min_sigma_from_col(grs_popt_col)   # gRS used the standard 0.25 floor

# RS-tuned = gRS significant AND it beats BOTH the OF Gaussian and the 2D Gaussian.
# Requiring gRS > g2d excludes cells with conjunctive RS×OF coding (product / 2D), which
# the 2D model fits better. of_tuned is defined symmetrically.
neurons_df["rs_tuned"] = (
    neurons_df[f"{grs_rsq_col}_sig"]
    & (neurons_df[grs_rsq_col] > neurons_df[gof_rsq_col])
    & (neurons_df[grs_rsq_col] > neurons_df[g2d_rsq_col])
)
neurons_df["of_tuned"] = (
    neurons_df[f"{gof_rsq_col}_sig"]
    & (neurons_df[gof_rsq_col] > neurons_df[grs_rsq_col])
    & (neurons_df[gof_rsq_col] > neurons_df[g2d_rsq_col])
)

# preferred RS (cm/s) and shape class from the gRS peak location
neurons_df["pref_rs_cms"] = neurons_df["preferred_RS_closedloop_grs_treadmill_trial_average"] * 100
def _rs_shape(p):
    if not np.isfinite(p):
        return None
    if p <= RS_LO_CMS:
        return "decreasing"
    if p >= RS_HI_CMS:
        return "increasing"
    return "peaked"
neurons_df["rs_shape"] = neurons_df["pref_rs_cms"].apply(_rs_shape)

shapes = ["increasing", "peaked", "decreasing"]
base = neurons_df[neurons_df[grs_popt_col].notna()]   # all cells with a valid gRS fit
rt = base[base["rs_tuned"]]                            # the RS-tuned subset

print(f"{len(base)} cells with valid gRS fit; {len(rt)} RS-tuned "
      f"({len(rt) / len(base) * 100:.1f}%)")
for layer in layers:
    g = base[base["layer"] == layer]
    print(f"  {layer}: RS-tuned={g['rs_tuned'].mean() * 100:.1f}%  "
          f"OF-tuned={g['of_tuned'].mean() * 100:.1f}%  (n={len(g)})")

In [ ]:
# --- RS Plot 1: prevalence of RS-tuned / OF-tuned / g2d-best / unfit cells per layer ---
# base = ALL neurons. Four mutually-exclusive classes that partition every neuron,
# expressed as a fraction of all neurons in each layer:
#   RS-tuned : gRS significant & gRS > gOF & gRS > g2d
#   OF-tuned : gOF significant & gOF > gRS & gOF > g2d
#   g2d-best : g2d significant & g2d > gRS & g2d > gOF  (better fit than either 1D model)
#   unfit    : none of the above -> not well fitted by any of the 3 models
grs_rsq_col = model_rsq_cols["grs_treadmill"]
gof_rsq_col = model_rsq_cols["gof_treadmill"]
g2d_rsq_col = model_rsq_cols["g2d_treadmill"]
fitted = (base[f"{grs_rsq_col}_sig"] | base[f"{gof_rsq_col}_sig"] | base[f"{g2d_rsq_col}_sig"])
base = neurons_df.copy()
base["rs_tuned"] = base[f"{grs_rsq_col}_sig"] 
base["of_tuned"] = base[f"{gof_rsq_col}_sig"]
base["g2d_best"] = base[f"{g2d_rsq_col}_sig"] 
base["unfit"] = ~fitted

cat_cols = ["rs_tuned", "of_tuned", "g2d_best", "unfit"]

rows = []
for layer in layers:
    g = base[base["layer"] == layer]
    rows.append(dict(layer=layer, n=len(g), **{c: g[c].mean() for c in cat_cols}))
fr = pd.DataFrame(rows).set_index("layer")

# per-session, per-layer fraction of each category (used for the scatter + stats below)
sess_rows = []
for (sess, layer), g in base.groupby(["session", "layer"]):
    if len(g) == 0:
        continue
    sess_rows.append(dict(session=sess, layer=layer, n=len(g),
                          **{c: g[c].mean() for c in cat_cols}))
sess_df = pd.DataFrame(sess_rows)

fig, ax = plt.subplots(figsize=(8, 4))
# x = model class (RS / OF / g2d / unfit), color = layer
layer_colors = {"Layer 2/3": "#4477aa", "Layer 5": "#ee6677"}
plot_df = fr[cat_cols].T
plot_df.index = ["RS-tuned\n(gRS>gOF,g2d)",
                 "OF-tuned\n(gOF>gRS,g2d)",
                 "g2d-best\n(g2d>gRS,gOF)",
                 "not fit"]
plot_df.plot.bar(ax=ax, width=0.7, color=[layer_colors.get(l) for l in plot_df.columns])

# overlay one scatter point per session above each bar, same colour as the bar
rng = np.random.default_rng(0)
for layer, container in zip(plot_df.columns, ax.containers):
    pts = sess_df[sess_df["layer"] == layer]
    for col, bar in zip(cat_cols, container):
        xc = bar.get_x() + bar.get_width() / 2
        y = pts[col].dropna().values
        x = xc + rng.uniform(-bar.get_width() / 4, bar.get_width() / 4, size=len(y))
        ax.scatter(x, y, s=14, color=layer_colors[layer],
                   edgecolor="black", linewidth=0.4, zorder=3)

ax.set_ylabel("fraction of all cells")
ax.set_xlabel("")
ax.set_xticklabels(plot_df.index, rotation=0)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor=layer_colors[l], label=f"{l} (n={int(fr.loc[l, 'n'])})") for l in fr.index], title="layer")
plt.tight_layout()
plt.show()

print(fr[cat_cols].round(3).to_string())

# --- Per-session stats: is any category enriched in one layer? ---
# Compare the distribution of per-session fractions between layers (Mann-Whitney U, two-sided).
from scipy.stats import mannwhitneyu
l23 = sess_df[sess_df.layer == "Layer 2/3"]
l5 = sess_df[sess_df.layer == "Layer 5"]

# run the test for every category first, so we know how many tests to Bonferroni-correct
res = []
for c in cat_cols:
    a, b = l23[c].dropna(), l5[c].dropna()
    if len(a) < 2 or len(b) < 2:
        res.append(dict(cat=c, a=a, b=b, U=None, p=None))
        continue
    U, pval = mannwhitneyu(a, b, alternative="two-sided")
    res.append(dict(cat=c, a=a, b=b, U=U, p=pval))

n_tests = sum(r["p"] is not None for r in res)

def stars(p_corr):
    if p_corr < 0.001: return "***"
    if p_corr < 0.01:  return "**"
    if p_corr < 0.05:  return "*"
    return "ns"

print("\nPer-session category fractions: Layer 2/3 vs Layer 5 (Mann-Whitney U, two-sided)")
print(f"Bonferroni correction over {n_tests} tests; stars: *** p<0.001  ** p<0.01  * p<0.05")
print(f"{'category':<10} {'n_sess_L23':>10} {'n_sess_L5':>9} {'med_L23':>8} {'med_L5':>8} {'higher_in':>11} {'U':>8} {'p':>9} {'p_bonf':>9} {'sig':>4}")
for r in res:
    c, a, b = r["cat"], r["a"], r["b"]
    if r["p"] is None:
        print(f"{c:<10} {len(a):>10} {len(b):>9}  (too few sessions for a test)")
        continue
    higher = "Layer 5" if b.median() > a.median() else "Layer 2/3"
    p_bonf = min(r["p"] * n_tests, 1.0)
    print(f"{c:<10} {len(a):>10} {len(b):>9} {a.median():>8.3f} {b.median():>8.3f} {higher:>11} {r['U']:>8.0f} {r['p']:>9.2e} {p_bonf:>9.2e} {stars(p_bonf):>4}")

In [ ]:
# Per-session, per-layer mean model r^2 (depth-tuned cells only).
# Defines avg_by_sess used by the comparison cells below.
neurons_df_sig = neurons_df[
    (neurons_df["depth_tuning_test_spearmanr_rval_closedloop_treadmill"] > 0.1)
    & (neurons_df["depth_tuning_test_spearmanr_pval_closedloop_treadmill"] < 0.05)
]
avg_by_sess = (
    neurons_df_sig.groupby(["session", "layer"]).mean(numeric_only=True).reset_index()
)

In [ ]:
l23_gration = avg_by_sess[avg_by_sess.layer == 'Layer 2/3'].rsof_test_rsq_closedloop_gratio_treadmill_trial_average
l23_dxrs = avg_by_sess[avg_by_sess.layer == 'Layer 2/3'].rsof_test_rsq_closedloop_g2mult_treadmill_trial_average

l5_gration = avg_by_sess[avg_by_sess.layer == 'Layer 5'].rsof_test_rsq_closedloop_gratio_treadmill_trial_average
l5_dxrs = avg_by_sess[avg_by_sess.layer == 'Layer 5'].rsof_test_rsq_closedloop_g2mult_treadmill_trial_average

print(np.nanmedian((l23_dxrs - l23_gration)/l23_gration))
print(np.nanmedian((l5_dxrs - l5_gration)/l5_gration))
plt.hist((l23_dxrs - l23_gration)/l23_gration,  alpha=0.5, bins=np.arange(0, 3, 0.1), label='Layer 2/3', color='C0')
_ = plt.hist((l5_dxrs - l5_gration)/l5_gration, alpha=0.5, bins=np.arange(0, 3, 0.1), label='Layer 5', color='C1')


In [ ]:
# --- RS Plot 2: the fitted gRS tuning curves themselves, normalised, by shape and layer ---
rs_grid = np.logspace(np.log10(1), np.log10(60), 60)   # display running-speed grid (cm/s)
logrs = np.log(rs_grid / 100)                          # fit space: log(RS in m/s)

def grs_curve(popt):
    la, x0, ls2, off = float(popt[0]), float(popt[1]), float(popt[2]), float(popt[3])
    sig2 = np.exp(ls2) + GRS_MIN_SIGMA
    return off + np.exp(la) * np.exp(-(logrs - x0) ** 2 / (2 * sig2))

colors = {"Layer 2/3": "#4477aa", "Layer 5": "#ee6677"}
rng = np.random.RandomState(0)
fig, axes = plt.subplots(1, len(shapes), figsize=(5 * len(shapes), 4), sharey=True)
for ax, shape in zip(np.atleast_1d(axes), shapes):
    for layer in layers:
        grp = rt[(rt["layer"] == layer) & (rt["rs_shape"] == shape)]
        curves = []
        for popt in grp[grs_popt_col]:
            c = grs_curve(np.asarray(popt, float))
            span = c.max() - c.min()
            if span > 0:
                curves.append((c - c.min()) / span)   # min-max normalise to compare shapes
        if not curves:
            continue
        curves = np.array(curves)
        for c in curves[rng.choice(len(curves), min(40, len(curves)), replace=False)]:
            ax.plot(rs_grid, c, color=colors[layer], alpha=0.08, lw=0.8)
        ax.plot(rs_grid, curves.mean(0), color=colors[layer], lw=2.5,
                label=f"{layer} (n={len(curves)})")
    ax.set_xscale("log")
    ax.set_xlabel("running speed (cm/s)")
    ax.set_title(shape)
    ax.legend(fontsize=8)
axes[0].set_ylabel("normalised dF/F")
fig.suptitle("Fitted 1D-RS tuning curves of RS-tuned cells")
plt.tight_layout()
plt.show()

In [ ]:
# --- RS Plot 3: tuning-shape composition per layer + chi-square test ---
cnt = pd.crosstab(rt["layer"], rt["rs_shape"]).reindex(columns=shapes, fill_value=0)
comp = cnt.div(cnt.sum(axis=1), axis=0).mul(100)

fig, ax = plt.subplots(figsize=(6, 4))
comp.plot.bar(ax=ax, width=0.75, color=["#228833", "#ccbb44", "#aa3377"])
ax.set_ylabel("% of RS-tuned cells")
ax.set_xlabel("")
ax.set_xticklabels([f"{l}\n(n={int(cnt.loc[l].sum())})" for l in comp.index], rotation=0)
ax.legend(title="RS tuning shape")
chi2, p, _, _ = chi2_contingency(cnt)
ax.set_title(f"RS tuning-shape composition by layer  (χ²={chi2:.0f}, p={p:.1e})")
plt.tight_layout()
plt.show()
print(comp.round(1).to_string())

In [ ]:
# --- RS Plot 4: preferred-RS distribution per layer (out-of-range grouped as low / high) ---
# Log-spaced bins centered on the stimulus grid [4, 8, 16, 32, 64] cm/s (factor-2 spacing).
# Bin edges are the geometric midpoints between grid points, so each bar is centered on
# its grid value. Out-of-range cells are pooled into a low (~2 cm/s) and a high (~128 cm/s)
# catch-all, i.e. the next octave below / above the grid; their boundaries are the
# midpoints sqrt(2*4)=2.83 and sqrt(64*128)=90.5 cm/s.
neurons_df["rs_tuned"] = (
    neurons_df[f"{grs_rsq_col}_sig"]
    #& (neurons_df[grs_rsq_col] > neurons_df[gof_rsq_col])
    #& (neurons_df[grs_rsq_col] > neurons_df[g2d_rsq_col])
)
rt = neurons_df[neurons_df["rs_tuned"]].copy()
RS_GRID_CMS = np.array([4, 8, 16, 32, 64], dtype=float)
inner_edges = np.sqrt(RS_GRID_CMS[:-1] * RS_GRID_CMS[1:])   # 5.66, 11.3, 22.6, 45.3
lo_edge = RS_GRID_CMS[0] / np.sqrt(2)                       # 2.83 -> below = low (~2)
hi_edge = RS_GRID_CMS[-1] * np.sqrt(2)                      # 90.5 -> above = high (~128)
edges = np.concatenate([[lo_edge], inner_edges, [hi_edge]])
centers = RS_GRID_CMS

xpos = np.arange(len(centers) + 2)       # [low] + in-range bins + [high]
width = 0.4
offsets = {"Layer 2/3": -width / 2, "Layer 5": +width / 2}
colors = {"Layer 2/3": "#4477aa", "Layer 5": "#ee6677"}

def _frac(p):
    """Percent of cells per bin: [low] + in-range histogram + [high]."""
    p = p.dropna()
    n = len(p)
    if n == 0:
        return None, 0
    low = int((p < lo_edge).sum())
    high = int((p >= hi_edge).sum())
    counts, _ = np.histogram(p[(p >= lo_edge) & (p < hi_edge)], bins=edges)
    return np.concatenate([[low], counts, [high]]) / n * 100, n

fig, ax = plt.subplots(figsize=(9, 4))
for layer in layers:
    sub = rt.loc[rt["layer"] == layer]
    frac, n = _frac(sub["pref_rs_cms"])
    ax.bar(xpos + offsets[layer], frac, width=width, color=colors[layer],
           label=f"{layer} (n={n})")
    # thin black line per session over the layer's bars
    for _sess, _g in sub.groupby("session"):
        sfrac, sn = _frac(_g["pref_rs_cms"])
        if sfrac is not None:
            ax.plot(xpos + offsets[layer], sfrac, 
                    marker='o',mec='k', color=colors[layer], lw=0.4, alpha=0.4)

# shade the out-of-range (low / high) categories
ax.axvspan(-0.5, 0.5, color="grey", alpha=0.12)
ax.axvspan(len(centers) + 0.5, len(centers) + 1.5, color="grey", alpha=0.12)

labels = ["low\n(≲2)"] + [f"{c:.0f}" for c in centers] + ["high\n(≳128)"]
ax.set_xticks(xpos)
ax.set_xticklabels(labels, fontsize=9)
ax.set_xlabel("preferred running speed (cm/s)  —  log bins centered on the 4–64 grid")
ax.set_ylabel("% of RS-tuned cells")
ax.legend()
ax.set_title("Preferred-RS distribution by layer (grid-centered log bins)")
plt.tight_layout()
plt.show()


In [ ]:
# --- OF Plot 4: preferred-OF distribution per layer (out-of-range grouped as low / high) ---
# Optic flow is sampled on a factor-4 grid [1, 4, 16, 64, 256, 1024] deg/s. Bin edges are the
# geometric midpoints (sqrt of adjacent centers), so each bar is centered on its grid value.
# preferred_OF is stored in rad/s (fit-native units) -> convert to deg/s with np.degrees,
# exactly as the viewer does (of_t = np.degrees(OF_stim)). Out-of-range cells (the gOF fit
# railed below the grid / above it) are pooled into a low (~0.25) and high (~4096) catch-all,
# the next factor-4 step beyond the grid; boundaries are sqrt(1/4)=0.5 and sqrt(1024*4096)=2048.
neurons_df["of_tuned"] = (
    neurons_df[f"{gof_rsq_col}_sig"]
    #& (neurons_df[gof_rsq_col] > neurons_df[grs_rsq_col])
    # & (neurons_df[gof_rsq_col] > neurons_df[g2d_rsq_col])
)
ot = neurons_df[neurons_df["of_tuned"]].copy()
ot["pref_of_degs"] = np.degrees(ot["preferred_OF_closedloop_gof_treadmill_trial_average"])

OF_GRID_DEGS = np.array([1, 4, 16, 64, 256, 1024], dtype=float)
inner_edges = np.sqrt(OF_GRID_DEGS[:-1] * OF_GRID_DEGS[1:])   # 2, 8, 32, 128, 512
lo_edge = OF_GRID_DEGS[0] / 2.0                               # 0.5 -> below = low (~0.25)
hi_edge = OF_GRID_DEGS[-1] * 2.0                              # 2048 -> above = high (~4096)
edges = np.concatenate([[lo_edge], inner_edges, [hi_edge]])
centers = OF_GRID_DEGS

xpos = np.arange(len(centers) + 2)       # [low] + in-range bins + [high]
width = 0.4
offsets = {"Layer 2/3": -width / 2, "Layer 5": +width / 2}
colors = {"Layer 2/3": "#4477aa", "Layer 5": "#ee6677"}

def _frac(p):
    """Percent of cells per bin: [low] + in-range histogram + [high]."""
    p = p.dropna()
    n = len(p)
    if n == 0:
        return None, 0
    low = int((p < lo_edge).sum())
    high = int((p >= hi_edge).sum())
    counts, _ = np.histogram(p[(p >= lo_edge) & (p < hi_edge)], bins=edges)
    return np.concatenate([[low], counts, [high]]) / n * 100, n

fig, ax = plt.subplots(figsize=(9, 4))
for layer in layers:
    sub = ot.loc[ot["layer"] == layer]
    frac, n = _frac(sub["pref_of_degs"])
    ax.bar(xpos + offsets[layer], frac, width=width, color=colors[layer],
           label=f"{layer} (n={n})")
    # thin black line per session over the layer's bars
    for _sess, _g in sub.groupby("session"):
        sfrac, sn = _frac(_g["pref_of_degs"])
        if sfrac is not None:
            ax.plot(xpos + offsets[layer], sfrac,
            marker='o',mec='k', color=colors[layer], lw=0.4, alpha=0.4)

# shade the out-of-range (low / high) categories
ax.axvspan(-0.5, 0.5, color="grey", alpha=0.12)
ax.axvspan(len(centers) + 0.5, len(centers) + 1.5, color="grey", alpha=0.12)

labels = ["low\n(≲0.25)"] + [f"{c:.0f}" for c in centers] + ["high\n(≳4096)"]
ax.set_xticks(xpos)
ax.set_xticklabels(labels, fontsize=9)
ax.set_xlabel("preferred optic flow (deg/s)  —  log bins centered on the 1–1024 grid")
ax.set_ylabel("% of OF-tuned cells")
ax.legend()
ax.set_title("Preferred-OF distribution by layer (grid-centered log bins)")
plt.tight_layout()
plt.show()


In [ ]:
# --- Depth Plot 4: preferred-depth distribution per layer (out-of-range grouped as low / high) ---

dt = neurons_df[(neurons_df.is_depth_neuron_treadmill)].copy()
# gDepth = the gaussian_ratio ("pure depth") model. Its preferred RS/OF ratio is stored as
# np.degrees(exp(popt[1])) = RS_m / OF_rad; in closed loop OF_rad = RS_m / depth_m, so this
# ratio is the preferred depth in metres -> *100 for cm.
gratio_ratio_col = "preferred_RSOFratio_closedloop_gratio_treadmill_trial_average"
dt["pref_depth_cm"] = dt[gratio_ratio_col] * 100
dt["pref_depth_cm"] = dt["pref_depth_cm"].where(np.isfinite(dt["pref_depth_cm"]))

DEPTH_GRID_CMS = np.unique(np.hstack(RS_GRID_CMS[None, :] / np.deg2rad(OF_GRID_DEGS)[:, None]))[1:-1:2]
inner_edges = np.sqrt(DEPTH_GRID_CMS[:-1] * DEPTH_GRID_CMS[1:])   # 7.07, 14.1, ...
lo_edge = DEPTH_GRID_CMS[0] / np.sqrt(2)                          # 3.54 -> below = low (~2.5)
hi_edge = DEPTH_GRID_CMS[-1] * np.sqrt(2)                         # 905  -> above = high (~1280)
edges = np.concatenate([[lo_edge], inner_edges, [hi_edge]])
centers = DEPTH_GRID_CMS

xpos = np.arange(len(centers) + 2)       # [low] + in-range bins + [high]
width = 0.4
offsets = {"Layer 2/3": -width / 2, "Layer 5": +width / 2}
colors = {"Layer 2/3": "#4477aa", "Layer 5": "#ee6677"}

def _frac(p):
    """Percent of cells per bin: [low] + in-range histogram + [high]."""
    p = p.dropna()
    n = len(p)
    if n == 0:
        return None, 0
    low = int((p < lo_edge).sum())
    high = int((p >= hi_edge).sum())
    counts, _ = np.histogram(p[(p >= lo_edge) & (p < hi_edge)], bins=edges)
    return np.concatenate([[low], counts, [high]]) / n * 100, n

fig, ax = plt.subplots(figsize=(9, 4))
for layer in layers:
    sub = dt.loc[dt["layer"] == layer]
    frac, n = _frac(sub["pref_depth_cm"])
    ax.bar(xpos + offsets[layer], frac, width=width, color=colors[layer],
           label=f"{layer} (n={n})")
    # thin black line per session over the layer's bars
    for _sess, _g in sub.groupby("session"):
        sfrac, sn = _frac(_g["pref_depth_cm"])
        if sfrac is not None:
            ax.plot(xpos + offsets[layer], sfrac, 
                    marker='o',mec='k', color=colors[layer], lw=0.4, alpha=0.4)

# shade the out-of-range (low / high) categories
ax.axvspan(-0.5, 0.5, color="grey", alpha=0.12)
ax.axvspan(len(centers) + 0.5, len(centers) + 1.5, color="grey", alpha=0.12)

labels = [f"low\n(≲{lo_edge:.1f})"] + [f"{c:.1f}" for c in centers] + [f"high\n(≳{hi_edge:.0f})"]
ax.set_xticks(xpos)
ax.set_xticklabels(labels, fontsize=9, rotation=90)
ax.set_xlabel("preferred virtual depth (cm)")
ax.set_ylabel("% of depth tuned cells")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
d = np.sort(np.unique(np.hstack(RS_GRID_CMS[None, :] / np.deg2rad(OF_GRID_DEGS)[:, None])))
d[1:]/d[:-1]

In [ ]:
# --- ROI browser: RS/OF trial-average matrix + gRS/gOF/g2d/depth fit matrices + scatters ----
# Top row : data matrix, then the gRS, gOF, g2d and gratio("pure depth") fit matrices.
# Bottom  : OF / RS / depth scatters, each with the g2d fit (blue) and the relevant 1D model
#           (red: gRS on RS, gOF on OF, gratio on depth). popt is stored in RS=m/s, OF=deg/s
#           (log space) so the model funcs are called directly; of_t is already deg/s.
#           In closed loop OF_rad = RS_m/depth_m, so the gratio ratio RS_m/OF_deg = depth_m*pi/180;
#           the g2d depth curve fixes RS at its preferred value and lets OF follow depth.

from cottage_analysis.pipelines import pipeline_utils
from cottage_analysis.plotting import rsof_plots
from cottage_analysis.analysis import fit_gaussian_blob
from ipywidgets import interact, IntSlider

grs_rsq_col    = model_rsq_cols["grs_treadmill"]
gof_rsq_col    = model_rsq_cols["gof_treadmill"]
g2d_rsq_col    = model_rsq_cols["g2d_treadmill"]
gratio_rsq_col = model_rsq_cols["gratio_treadmill"]

# variance floors each fit was run with (recovered from the column tags; all 0.25 here)
GOF_MIN_SIGMA    = min_sigma_from_col("rsof_popt_closedloop_gof_treadmill_trial_average")
G2D_MIN_SIGMA    = min_sigma_from_col("rsof_popt_closedloop_g2d_treadmill_trial_average")
GRATIO_MIN_SIGMA = min_sigma_from_col("rsof_popt_closedloop_gratio_treadmill_trial_average")

# ROIs to browse, grouped per session (start near preferred RS = 8 cm/s).
roi_table = dt[np.isfinite(dt["pref_rs_cms"])].copy()
# Filter layer 5
roi_table = roi_table[roi_table["layer"] == "Layer 5"]
# only g2d rsq > 0.2
roi_table = roi_table[roi_table[g2d_rsq_col] > 0.2]
roi_table = roi_table.sort_values(["session", "pref_rs_cms"]).reset_index(drop=True)
print(f"{len(roi_table)} RS-tuned ROIs across {roi_table['session'].nunique()} sessions")

# session -> project map (built once so we don't hit flexilims per ROI)
session_project = {}
for _proj in PROJECTS:
    _fs = flz.get_flexilims_session(project_id=_proj)
    for _name in flz.get_entities(datatype="session", flexilims_session=_fs).name:
        session_project.setdefault(_name, _proj)

# Cache only the most recently loaded session's data.
_session_cache = {}

def get_session_data(sess):
    """Load (and cache) trials + log-grid bins for one session. Keeps only the last."""
    if sess in _session_cache:
        return _session_cache[sess]
    photodiode_protocol = 2 if (("PZAH6.4b" in sess) or ("PZAG3.4f" in sess)) else 5
    _, _, _, tdf = pipeline_utils.load_session(
        project=session_project[sess], session_name=sess,
        photodiode_protocol=photodiode_protocol, regenerate_frames=False,
        filter_datasets=dict(annotated=True), protocol_base="SpheresTubeMotor",
    )
    tdf = tdf[~tdf.recording_name.str.contains("multidepth")]
    tdf = tdf[tdf.closed_loop == 1].copy()

    motor_speeds = np.round(np.unique(tdf.MotorSpeed_stim.map(np.nanmedian)))
    ms_log = np.log2(motor_speeds); rs_bw = np.median(np.diff(ms_log))
    rs_bins = 2 ** (np.arange(ms_log[0] - rs_bw * 1.5, ms_log[-1] + rs_bw * 2, rs_bw))
    rs_bins = np.insert(rs_bins, 0, 0)
    of_speeds = np.round(np.unique(tdf.expected_optic_flow_stim.map(np.nanmedian)))
    of_log = np.log2(of_speeds); of_bw = np.median(np.diff(of_log))
    of_bins = 2 ** (np.arange(of_log[0] - of_bw * 0.5, of_log[-1] + of_bw, of_bw))
    of_bins = np.insert(of_bins, 0, 0)
    rs_mid = np.diff(np.log2(rs_bins[1:])) / 2 + np.log2(rs_bins[1:])[:-1]
    of_mid = np.diff(np.log2(of_bins[1:])) / 2 + np.log2(of_bins[1:])[:-1]
    tick_dict = dict(
        rs_tick_select=rs_mid, rs_tick_values=(2 ** rs_mid).astype(int),
        of_tick_select=of_mid, of_tick_values=(2 ** of_mid).astype(int),
    )
    data = dict(trials_df=tdf, rs_bins=rs_bins, of_bins=of_bins, tick_dict=tick_dict)
    _session_cache.clear()          # cache the last session only
    _session_cache[sess] = data
    return data

def plot_roi(row):
    sess, roi = row["session"], int(row["roi"])
    data = get_session_data(sess)
    tdf = data["trials_df"]
    range_kwargs = dict(log_range={"log_base": 2}, rs_bins=data["rs_bins"],
                        of_bins=data["of_bins"], tick_dict=data["tick_dict"])
    fontsize_dict = {"title": 12, "label": 11, "tick": 9, "legend": 8}

    # collapse each trial to its mean -> a genuine trial-average matrix when binned
    def _med(a):
        return np.array([np.nanmedian(np.asarray(a, dtype=float))])
    ta = tdf.copy()
    ta["RS_stim"] = ta["RS_stim"].apply(_med)
    ta["OF_stim"] = ta["OF_stim"].apply(_med)
    ta["acceleration_ratio_max_stim"] = ta["acceleration_ratio_max_stim"].apply(_med)
    if "max_abs_rs2motor_diff_ratio_stim" in ta.columns:
        ta["max_abs_rs2motor_diff_ratio_stim"] = ta["max_abs_rs2motor_diff_ratio_stim"].apply(_med)
    ta["dff_stim"] = ta["dff_stim"].apply(lambda d: np.nanmean(np.asarray(d), axis=0, keepdims=True))

    # per-trial scalars for the scatters
    rs_t = tdf["RS_stim"].apply(np.nanmedian).values * 100.0
    of_t = np.degrees(tdf["OF_stim"].apply(np.nanmedian).values)
    depth_t = tdf["depth"].values * 100.0
    dff_t = tdf["dff_stim"].apply(lambda d: np.nanmean(np.asarray(d)[:, roi])).values
    keep = np.isfinite(dff_t) & np.isfinite(rs_t)
    if "max_abs_rs2motor_diff_ratio_stim" in tdf.columns:
        rs2m = tdf["max_abs_rs2motor_diff_ratio_stim"].apply(np.nanmedian).values
        keep = keep & (rs2m < 0.3)
    rs_t, of_t, depth_t, dff_t = rs_t[keep], of_t[keep], depth_t[keep], dff_t[keep]

    # ---- fit parameters (popt in RS=m/s, OF=deg/s log space) ----
    popt_grs    = row["rsof_popt_closedloop_grs_treadmill_trial_average"]
    popt_gof    = row["rsof_popt_closedloop_gof_treadmill_trial_average"]
    popt_g2d    = row["rsof_popt_closedloop_g2d_treadmill_trial_average"]
    popt_gratio = row["rsof_popt_closedloop_gratio_treadmill_trial_average"]
    pref_rs_g2d_ms   = np.exp(popt_g2d[1])    # g2d preferred RS (m/s)
    pref_of_g2d_degs = np.exp(popt_g2d[2])    # g2d preferred OF (deg/s)
    # clamp the preferred RS/OF used for the g2d overlays to the presented range
    pref_rs_g2d_ms   = np.clip(pref_rs_g2d_ms, 4 / 100, 64 / 100)      # RS in 4-64 cm/s
    pref_of_g2d_degs = np.clip(pref_of_g2d_degs, 1, 1024)             # OF in 1-1024 deg/s

    # axis support lines for the overlays
    rs_line    = np.logspace(np.log10(max(rs_t.min(), 1)), np.log10(rs_t.max()), 200)
    of_line    = np.logspace(np.log10(max(of_t[of_t > 0].min(), 1e-2)),
                             np.log10(of_t.max()), 200)
    depth_line = np.logspace(np.log10(depth_t.min()), np.log10(depth_t.max()), 200)

    # 1D model curves
    grs_curve = fit_gaussian_blob.gaussian_1d(
        np.log(rs_line / 100), *popt_grs, min_sigma=GRS_MIN_SIGMA)
    gof_curve = fit_gaussian_blob.gaussian_1d(
        np.log(of_line), *popt_gof, min_sigma=GOF_MIN_SIGMA)
    # gratio is a 1D Gaussian on log(RS_m/OF_deg) = log(depth_m * pi/180)
    gratio_curve = fit_gaussian_blob.gaussian_1d(
        np.log(depth_line / 100 * np.pi / 180), *popt_gratio, min_sigma=GRATIO_MIN_SIGMA)

    # g2d slices through the g2d peak (fix the other variable / let OF follow depth)
    g2d_rs_curve = fit_gaussian_blob.gaussian_2d(
        (np.log(rs_line / 100), np.full_like(rs_line, np.log(pref_of_g2d_degs))),
        *popt_g2d, min_sigma=G2D_MIN_SIGMA)
    g2d_of_curve = fit_gaussian_blob.gaussian_2d(
        (np.full_like(of_line, np.log(pref_rs_g2d_ms)), np.log(of_line)),
        *popt_g2d, min_sigma=G2D_MIN_SIGMA)
    # depth locus: RS fixed at preferred, OF = RS/depth (deg/s)
    of_from_depth = np.degrees(pref_rs_g2d_ms / (depth_line / 100))
    g2d_dep_curve = fit_gaussian_blob.gaussian_2d(
        (np.full_like(depth_line, np.log(pref_rs_g2d_ms)), np.log(of_from_depth)),
        *popt_g2d, min_sigma=G2D_MIN_SIGMA)

    # ---- figure: 5 matrices on top, 3 scatters below ----
    fig = plt.figure(figsize=(20, 8))
    ax_data  = plt.subplot2grid((2, 5), (0, 0), fig=fig)
    ax_grs   = plt.subplot2grid((2, 5), (0, 1), fig=fig)
    ax_gof   = plt.subplot2grid((2, 5), (0, 2), fig=fig)
    ax_g2d   = plt.subplot2grid((2, 5), (0, 3), fig=fig)
    ax_gdep  = plt.subplot2grid((2, 5), (0, 4), fig=fig)
    ax_of    = plt.subplot2grid((2, 3), (1, 0), fig=fig)
    ax_rs    = plt.subplot2grid((2, 3), (1, 1), fig=fig)
    ax_depth = plt.subplot2grid((2, 3), (1, 2), fig=fig)

    vmin, vmax, _ = rsof_plots.plot_RS_OF_matrix(
        trials_df=ta, roi=roi, is_closed_loop=1, max_abs_rs2motor_diff_ratio=0.3,
        ax=ax_data, fontsize_dict=fontsize_dict, return_matrix=True,
        title="Trial-average response", cbar_width=0.01, vmin=0, **range_kwargs)

    ndf_sess = neurons_df[neurons_df.session == sess]
    for ax, model, ms, label, rsq_col in [
        (ax_grs,  "grs",    GRS_MIN_SIGMA,    "gRS fit",            grs_rsq_col),
        (ax_gof,  "gof",    GOF_MIN_SIGMA,    "gOF fit",            gof_rsq_col),
        (ax_g2d,  "g2d",    G2D_MIN_SIGMA,    "g2d fit",            g2d_rsq_col),
        (ax_gdep, "gratio", GRATIO_MIN_SIGMA, "depth (gratio) fit", gratio_rsq_col),
    ]:
        rsof_plots.plot_RS_OF_fit(
            neurons_df=ndf_sess, roi=roi, model=model, sfx="_treadmill_trial_average",
            model_label=label, min_sigma=ms, vmin=0, vmax=vmax, ax=ax, label_r2=False,
            fontsize_dict=fontsize_dict, cbar_width=0.01, **range_kwargs)
        ax.text(0.05, 0.92, f"$R^2$={row[rsq_col]:.2f}", transform=ax.transAxes,
                fontsize=fontsize_dict["tick"])

    for ax, x, xlab in [(ax_of, of_t, "Optic flow (deg/s)"),
                        (ax_rs, rs_t, "Running speed (cm/s)"),
                        (ax_depth, depth_t, "Virtual depth (cm)")]:
        ax.scatter(x, dff_t, s=14, alpha=0.4, color="#444444", edgecolor="none")
        ax.set_xscale("log")
        ax.set_xlabel(xlab, fontsize=fontsize_dict["label"])
        ax.set_ylabel("trial mean dF/F", fontsize=fontsize_dict["label"])
        ax.spines[["top", "right"]].set_visible(False)

    # RS scatter: gRS (red) + g2d slice at preferred OF (blue)
    ax_rs.plot(rs_line, grs_curve, color="#ee6677", lw=2, label="gRS fit")
    ax_rs.plot(rs_line, g2d_rs_curve, color="#4477aa", lw=2, label="g2d (@pref OF)")
    ax_rs.axvline(row["pref_rs_cms"], color="#ee6677", ls="--", lw=1, alpha=0.7)
    ax_rs.legend(fontsize=fontsize_dict["legend"], frameon=False)

    # OF scatter: gOF (red) + g2d slice at preferred RS (blue)
    ax_of.plot(of_line, gof_curve, color="#ee6677", lw=2, label="gOF fit")
    ax_of.plot(of_line, g2d_of_curve, color="#4477aa", lw=2, label="g2d (@pref RS)")
    ax_of.axvline(pref_of_g2d_degs, color="#4477aa", ls="--", lw=1, alpha=0.7)
    ax_of.legend(fontsize=fontsize_dict["legend"], frameon=False)

    # depth scatter: gratio "pure depth" (red) + g2d depth-locus at preferred RS (blue)
    ax_depth.plot(depth_line, gratio_curve, color="#ee6677", lw=2, label="depth (gratio) fit")
    ax_depth.plot(depth_line, g2d_dep_curve, color="#4477aa", lw=2, label="g2d (@pref RS)")
    ax_depth.legend(fontsize=fontsize_dict["legend"], frameon=False)

    fig.suptitle(
        f"{sess}  roi {roi}  —  {row['layer']}  —  "
        f"preferred RS = {row['pref_rs_cms']:.0f} cm/s  (gRS $R^2$={row[grs_rsq_col]:.2f})",
        fontsize=fontsize_dict["title"])
    plt.show()

def browse(i):
    plot_roi(roi_table.iloc[i])

# start at the ROI whose preferred RS is closest to 8 cm/s
start = int((roi_table["pref_rs_cms"] - 8).abs().idxmin())
interact(browse, i=IntSlider(value=start, min=0, max=len(roi_table) - 1, step=1,
                             description="ROI #", continuous_update=False));


In [ ]:
61/(2**np.arange(1, 6))